# Lab 19: CycleGAN — Unpaired Domain Translation

**Goal:** Build CycleGAN from scratch and see cycle consistency in action.

**Task:** Translate between two MNIST domains *without paired data*:
- **Domain A:** Normal MNIST digits (white strokes on a black background)
- **Domain B:** Inverted + thickened digits (black, bolder strokes on a white background)

What we'll build:
1. Create two unpaired image domains from MNIST
2. ResNet-based generators (G_AB and G_BA)
3. PatchGAN discriminators (D_A and D_B)
4. Cycle consistency loss + LSGAN loss
5. Train and watch the cycle reconstruct originals
6. Experiment: what happens WITHOUT cycle loss?

**Notation (Unit 5 · L19):** there is no target image for a given input — x and y are just images from the two domains. The paper calls the generators G: X→Y and F: Y→X; here they are `G_AB` and `G_BA`.

**Runtime (measured on an Apple-silicon laptop, two training runs):** full data is slow — about 21 min/epoch on a laptop CPU, 1.2 min/epoch on `mps`. So the notebook picks the size for you: `mps` uses 10,000 images per domain for 30 epochs (≈ 25 min for both runs); CPU uses 3,000 per domain for 10 epochs (≈ 45 min); a CUDA GPU uses everything.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset, Subset
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(42)

# Images per domain and epochs, chosen for the device (see runtime note above)
PER_DOMAIN = {"cuda": None, "mps": 10_000, "cpu": 3_000}[device.type]
EPOCHS     = {"cuda": 30,   "mps": 30,     "cpu": 10}[device.type]
print(f"Using: {device}")

## 1. Create Two Unpaired Domains

We split MNIST into two non-overlapping halves and apply different styles:
- **Domain A:** Normal digits (one random half of MNIST)
- **Domain B:** Inverted + thickened (the *other* half — different images)

The domains use *different images* — there are no pairs!

In [ ]:
def thicken(img, kernel_size=3):
    """Dilate digit strokes to make them thicker."""
    # img: (1, 28, 28) in [-1, 1]
    img_01 = (img + 1) / 2  # [0, 1]
    kernel = torch.ones(1, 1, kernel_size, kernel_size)
    thick = F.conv2d(img_01.unsqueeze(0), kernel, padding=kernel_size//2)
    thick = thick.squeeze(0).clamp(0, 1)
    return thick * 2 - 1  # back to [-1, 1]


def make_domain_b(img):
    """Transform to domain B: invert + thicken."""
    thick = thicken(img, kernel_size=3)
    inverted = -thick  # invert: white strokes on black → black strokes on white
    return inverted


class DomainDataset(Dataset):
    """Wraps a subset of MNIST with an optional transform."""
    def __init__(self, mnist, indices, domain_transform=None):
        self.mnist = mnist
        self.indices = indices
        self.domain_transform = domain_transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img, label = self.mnist[self.indices[idx]]
        if self.domain_transform:
            img = self.domain_transform(img)
        return img


# Load MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])
mnist = datasets.MNIST("data", train=True, download=True, transform=transform)

# Split into two non-overlapping halves
n = len(mnist)
perm = torch.randperm(n)
indices_A = perm[:n//2].tolist()  # one half → domain A
indices_B = perm[n//2:].tolist()  # other half → domain B
if PER_DOMAIN:
    indices_A, indices_B = indices_A[:PER_DOMAIN], indices_B[:PER_DOMAIN]

domain_A = DomainDataset(mnist, indices_A, domain_transform=None)
domain_B = DomainDataset(mnist, indices_B, domain_transform=make_domain_b)

loader_A = DataLoader(domain_A, batch_size=64, shuffle=True, drop_last=True)
loader_B = DataLoader(domain_B, batch_size=64, shuffle=True, drop_last=True)

print(f"Domain A (normal):   {len(domain_A)} images")
print(f"Domain B (inverted): {len(domain_B)} images")
print(f"No pairing between them!")

In [ ]:
# Visualize the two domains
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(8):
    axes[0, i].imshow((domain_A[i].squeeze() + 1)/2, cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow((domain_B[i].squeeze() + 1)/2, cmap="gray")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Domain A\n(normal)", fontsize=11, rotation=0, labelpad=50)
axes[1, 0].set_ylabel("Domain B\n(inverted)", fontsize=11, rotation=0, labelpad=50)
plt.suptitle("Two Unpaired Domains — Different Images, Different Styles", fontsize=13)
plt.tight_layout()
plt.show()

## 2. Residual Block

The building block of CycleGAN's generator: conv → norm → relu → conv → norm + skip.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.InstanceNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.InstanceNorm2d(channels),
        )

    def forward(self, x):
        return x + self.block(x)  # output = input + changes

# Test
rb = ResidualBlock(64)
t = torch.randn(2, 64, 14, 14)
print(f"ResidualBlock: {t.shape} -> {rb(t).shape} (same size!)")

## 3. ResNet Generator

Downsample → Residual Blocks → Upsample

For 28x28 MNIST we use a smaller version (4 res blocks instead of 9).

In [ ]:
class ResNetGenerator(nn.Module):
    def __init__(self, n_res_blocks=4):
        super().__init__()

        # Initial convolution
        self.initial = nn.Sequential(
            nn.Conv2d(1, 64, 7, padding=3, bias=False),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True),
        )

        # Downsample
        self.down = nn.Sequential(
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(128),
            nn.ReLU(inplace=True),
        )

        # Residual blocks (all at 14x14)
        res_blocks = [ResidualBlock(128) for _ in range(n_res_blocks)]
        self.res = nn.Sequential(*res_blocks)

        # Upsample
        self.up = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1,
                               output_padding=1, bias=False),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True),
        )

        # Output
        self.output = nn.Sequential(
            nn.Conv2d(64, 1, 7, padding=3),
            nn.Tanh(),
        )

    def forward(self, x):
        x = self.initial(x)  # (b, 64, 28, 28)
        x = self.down(x)     # (b, 128, 14, 14)
        x = self.res(x)      # (b, 128, 14, 14) — same size!
        x = self.up(x)       # (b, 64, 28, 28)
        return self.output(x) # (b, 1, 28, 28)


# Test
gen = ResNetGenerator().to(device)
test_img = torch.randn(2, 1, 28, 28, device=device)
print(f"Generator: {test_img.shape} -> {gen(test_img).shape}")
print(f"Parameters: {sum(p.numel() for p in gen.parameters()):,}")

## 4. PatchGAN Discriminator

Same idea as Pix2Pix, but sees only ONE image (not a pair).

In [ ]:
class PatchDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1),          # 28 -> 14
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64, 128, 4, 2, 1, bias=False),  # 14 -> 7
            nn.InstanceNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 1, 4, 1, 1),          # 7 -> 6
            # No activation — LSGAN uses raw scores
        )

    def forward(self, x):
        return self.model(x)  # (b, 1, 6, 6)

# Test
disc = PatchDiscriminator().to(device)
print(f"Discriminator: {test_img.shape} -> {disc(test_img).shape}")
print(f"Parameters: {sum(p.numel() for p in disc.parameters()):,}")

## 5. Replay Buffer

Store recent fakes so D doesn't forget old ones.

In [ ]:
class ReplayBuffer:
    def __init__(self, max_size=50):
        self.max_size = max_size
        self.buffer = []

    def push_and_pop(self, images):
        result = []
        for img in images:
            img = img.unsqueeze(0)
            if len(self.buffer) < self.max_size:
                self.buffer.append(img.clone())
                result.append(img)
            elif torch.rand(1).item() > 0.5:
                idx = torch.randint(0, self.max_size, (1,)).item()
                old = self.buffer[idx].clone()
                self.buffer[idx] = img.clone()
                result.append(old)
            else:
                result.append(img)
        return torch.cat(result, dim=0)

print("ReplayBuffer ready.")

## 6. CycleGAN Training

Four losses:
1. **GAN A→B:** fake B fools D_B
2. **GAN B→A:** fake A fools D_A
3. **Cycle:** A→B→A ≈ A  and  B→A→B ≈ B
4. **Identity:** G_AB(B) ≈ B  and  G_BA(A) ≈ A

Weights: λ_cyc = 10, λ_id = 0.5·λ_cyc = 5 — as on the L19 slides.

**Predict first:** which term will dominate the G loss early on — GAN or 10 × cycle?

In [ ]:
# Create all models
G_AB = ResNetGenerator().to(device)   # A → B (normal → inverted)
G_BA = ResNetGenerator().to(device)   # B → A (inverted → normal)
D_A  = PatchDiscriminator().to(device)  # "Is this a real domain A image?"
D_B  = PatchDiscriminator().to(device)  # "Is this a real domain B image?"

# Optimizers
opt_G = torch.optim.Adam(
    list(G_AB.parameters()) + list(G_BA.parameters()),
    lr=2e-4, betas=(0.5, 0.999)
)
opt_D_A = torch.optim.Adam(D_A.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D_B = torch.optim.Adam(D_B.parameters(), lr=2e-4, betas=(0.5, 0.999))

# Loss functions
l1_loss = nn.L1Loss()

# Replay buffers
buffer_A = ReplayBuffer(50)
buffer_B = ReplayBuffer(50)

# Weights
lambda_cyc = 10.0
lambda_id  = 5.0

epochs = EPOCHS
history = {'g': [], 'd_a': [], 'd_b': [], 'cyc': []}

print("Models created:")
print(f"  G_AB, G_BA: {sum(p.numel() for p in G_AB.parameters()):,} params each")
print(f"  D_A, D_B:   {sum(p.numel() for p in D_A.parameters()):,} params each")
print(f"\nLoss weights: cycle={lambda_cyc}, identity={lambda_id}")
print(f"Epochs: {epochs}")

In [ ]:
def lsgan_d_loss(real_scores, fake_scores):
    """LSGAN discriminator loss."""
    return 0.5 * (torch.mean((real_scores - 1)**2) + torch.mean(fake_scores**2))

def lsgan_g_loss(fake_scores):
    """LSGAN generator loss."""
    return torch.mean((fake_scores - 1)**2)

print("LSGAN loss functions ready.")

In [ ]:
for epoch in range(epochs):
    g_sum, da_sum, db_sum, cyc_sum = 0, 0, 0, 0
    n_batches = 0

    for real_A, real_B in zip(loader_A, loader_B):
        real_A = real_A.to(device)
        real_B = real_B.to(device)

        # =========== GENERATOR STEP ===========
        # Generate
        fake_B = G_AB(real_A)    # A → B
        fake_A = G_BA(real_B)    # B → A

        # GAN losses
        loss_gan_AB = lsgan_g_loss(D_B(fake_B))
        loss_gan_BA = lsgan_g_loss(D_A(fake_A))

        # Cycle losses
        rec_A = G_BA(fake_B)     # A → B → A
        rec_B = G_AB(fake_A)     # B → A → B
        loss_cyc = l1_loss(rec_A, real_A) + l1_loss(rec_B, real_B)

        # Identity losses
        id_A = G_BA(real_A)      # A through B→A gen (should be unchanged)
        id_B = G_AB(real_B)      # B through A→B gen (should be unchanged)
        loss_id = l1_loss(id_A, real_A) + l1_loss(id_B, real_B)

        # Total G loss
        loss_G = (loss_gan_AB + loss_gan_BA
                  + lambda_cyc * loss_cyc
                  + lambda_id * loss_id)

        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()

        # =========== DISCRIMINATOR A STEP ===========
        fake_A_buf = buffer_A.push_and_pop(fake_A.detach())
        loss_D_A = lsgan_d_loss(D_A(real_A), D_A(fake_A_buf))

        opt_D_A.zero_grad()
        loss_D_A.backward()
        opt_D_A.step()

        # =========== DISCRIMINATOR B STEP ===========
        fake_B_buf = buffer_B.push_and_pop(fake_B.detach())
        loss_D_B = lsgan_d_loss(D_B(real_B), D_B(fake_B_buf))

        opt_D_B.zero_grad()
        loss_D_B.backward()
        opt_D_B.step()

        g_sum += loss_G.item()
        da_sum += loss_D_A.item()
        db_sum += loss_D_B.item()
        cyc_sum += loss_cyc.item()
        n_batches += 1

    history['g'].append(g_sum / n_batches)
    history['d_a'].append(da_sum / n_batches)
    history['d_b'].append(db_sum / n_batches)
    history['cyc'].append(cyc_sum / n_batches)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} | G: {history['g'][-1]:.3f} "
              f"| D_A: {history['d_a'][-1]:.3f} | D_B: {history['d_b'][-1]:.3f} "
              f"| Cycle: {history['cyc'][-1]:.3f}")

print("\nTraining complete!")

In [ ]:
# Loss curves
fig, axes = plt.subplots(1, 2, figsize=(12, 3))

axes[0].plot(history['g'], label='G total', alpha=0.8)
axes[0].plot(history['d_a'], label='D_A', alpha=0.8)
axes[0].plot(history['d_b'], label='D_B', alpha=0.8)
axes[0].set_title('GAN Losses')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['cyc'], label='Cycle loss', color='green', alpha=0.8)
axes[1].set_title('Cycle Consistency Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

for ax in axes:
    ax.set_xlabel('Epoch')

plt.tight_layout()
plt.show()
print("Cycle loss should decrease — the round-trip gets more accurate.")

## 7. Results: Translation + Cycle Reconstruction

The key test: A→B→A and B→A→B. The reconstructed image should match the original.

In [ ]:
G_AB.eval()
G_BA.eval()

n_show = 6

# Get some test images
test_A = torch.stack([domain_A[i] for i in range(n_show)]).to(device)
test_B = torch.stack([domain_B[i] for i in range(n_show)]).to(device)

with torch.no_grad():
    fake_B = G_AB(test_A)    # A → B
    rec_A  = G_BA(fake_B)    # A → B → A
    fake_A = G_BA(test_B)    # B → A
    rec_B  = G_AB(fake_A)    # B → A → B

# --- Plot A → B → A ---
fig, axes = plt.subplots(3, n_show, figsize=(12, 5))
for i in range(n_show):
    axes[0, i].imshow((test_A[i].cpu().squeeze() + 1)/2, cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow((fake_B[i].cpu().squeeze() + 1)/2, cmap="gray")
    axes[1, i].axis("off")
    axes[2, i].imshow((rec_A[i].cpu().squeeze() + 1)/2, cmap="gray")
    axes[2, i].axis("off")

axes[0, 0].set_ylabel("Real A", fontsize=11, rotation=0, labelpad=45)
axes[1, 0].set_ylabel("Fake B\n(A→B)", fontsize=11, rotation=0, labelpad=45)
axes[2, 0].set_ylabel("Recon A\n(A→B→A)", fontsize=11, rotation=0, labelpad=45)
plt.suptitle("Forward Cycle: A → B → A", fontsize=14)
plt.tight_layout()
plt.show()

# --- Plot B → A → B ---
fig, axes = plt.subplots(3, n_show, figsize=(12, 5))
for i in range(n_show):
    axes[0, i].imshow((test_B[i].cpu().squeeze() + 1)/2, cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow((fake_A[i].cpu().squeeze() + 1)/2, cmap="gray")
    axes[1, i].axis("off")
    axes[2, i].imshow((rec_B[i].cpu().squeeze() + 1)/2, cmap="gray")
    axes[2, i].axis("off")

axes[0, 0].set_ylabel("Real B", fontsize=11, rotation=0, labelpad=45)
axes[1, 0].set_ylabel("Fake A\n(B→A)", fontsize=11, rotation=0, labelpad=45)
axes[2, 0].set_ylabel("Recon B\n(B→A→B)", fontsize=11, rotation=0, labelpad=45)
plt.suptitle("Backward Cycle: B → A → B", fontsize=14)
plt.tight_layout()
plt.show()

# Reconstruction quality
rec_error_A = F.l1_loss(rec_A, test_A).item()
rec_error_B = F.l1_loss(rec_B, test_B).item()
print(f"Reconstruction L1 error (A→B→A): {rec_error_A:.4f}")
print(f"Reconstruction L1 error (B→A→B): {rec_error_B:.4f}")
print("\nLower = better cycle consistency. Row 3 should match row 1.")

## 8. Experiment 1: Without Cycle Loss

What happens if we remove cycle consistency? Nothing in the loss ties the output to the input any more.

**Predict first:** will A→B→A still look like the original digit?

In [ ]:
# Train WITHOUT cycle loss (GAN loss only)
G_AB_nocyc = ResNetGenerator().to(device)
G_BA_nocyc = ResNetGenerator().to(device)
D_A_nocyc  = PatchDiscriminator().to(device)
D_B_nocyc  = PatchDiscriminator().to(device)

opt_G_nc = torch.optim.Adam(
    list(G_AB_nocyc.parameters()) + list(G_BA_nocyc.parameters()),
    lr=2e-4, betas=(0.5, 0.999))
opt_DA_nc = torch.optim.Adam(D_A_nocyc.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_DB_nc = torch.optim.Adam(D_B_nocyc.parameters(), lr=2e-4, betas=(0.5, 0.999))

print("Training WITHOUT cycle loss (GAN only)...")
for epoch in range(epochs):
    for real_A, real_B in zip(loader_A, loader_B):
        real_A, real_B = real_A.to(device), real_B.to(device)

        fake_B = G_AB_nocyc(real_A)
        fake_A = G_BA_nocyc(real_B)

        # G: GAN loss ONLY (no cycle, no identity)
        loss_G = lsgan_g_loss(D_B_nocyc(fake_B)) + lsgan_g_loss(D_A_nocyc(fake_A))
        opt_G_nc.zero_grad(); loss_G.backward(); opt_G_nc.step()

        # D_A
        loss_DA = lsgan_d_loss(D_A_nocyc(real_A), D_A_nocyc(fake_A.detach()))
        opt_DA_nc.zero_grad(); loss_DA.backward(); opt_DA_nc.step()

        # D_B
        loss_DB = lsgan_d_loss(D_B_nocyc(real_B), D_B_nocyc(fake_B.detach()))
        opt_DB_nc.zero_grad(); loss_DB.backward(); opt_DB_nc.step()

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:3d}/{epochs}")

print("Done!")

In [ ]:
# Compare: with cycle vs without cycle
G_AB_nocyc.eval()
G_BA_nocyc.eval()

with torch.no_grad():
    fake_B_cyc   = G_AB(test_A).cpu()
    rec_A_cyc    = G_BA(G_AB(test_A)).cpu()
    fake_B_nocyc = G_AB_nocyc(test_A).cpu()
    rec_A_nocyc  = G_BA_nocyc(G_AB_nocyc(test_A)).cpu()

n_show = 6
fig, axes = plt.subplots(5, n_show, figsize=(12, 8))

for i in range(n_show):
    axes[0, i].imshow((test_A[i].cpu().squeeze() + 1)/2, cmap="gray")
    axes[0, i].axis("off")

    axes[1, i].imshow((fake_B_cyc[i].squeeze() + 1)/2, cmap="gray")
    axes[1, i].axis("off")

    axes[2, i].imshow((rec_A_cyc[i].squeeze() + 1)/2, cmap="gray")
    axes[2, i].axis("off")

    axes[3, i].imshow((fake_B_nocyc[i].squeeze() + 1)/2, cmap="gray")
    axes[3, i].axis("off")

    axes[4, i].imshow((rec_A_nocyc[i].squeeze() + 1)/2, cmap="gray")
    axes[4, i].axis("off")

axes[0, 0].set_ylabel("Real A", fontsize=10, rotation=0, labelpad=55)
axes[1, 0].set_ylabel("WITH cycle\nA→B", fontsize=10, rotation=0, labelpad=55)
axes[2, 0].set_ylabel("WITH cycle\nA→B→A", fontsize=10, rotation=0, labelpad=55)
axes[3, 0].set_ylabel("NO cycle\nA→B", fontsize=10, rotation=0, labelpad=55)
axes[4, 0].set_ylabel("NO cycle\nA→B→A", fontsize=10, rotation=0, labelpad=55)

plt.suptitle("WITH Cycle Loss vs WITHOUT: Does Content Survive?", fontsize=13)
plt.tight_layout()
plt.show()

rec_cyc   = F.l1_loss(rec_A_cyc, test_A.cpu()).item()
rec_nocyc = F.l1_loss(rec_A_nocyc, test_A.cpu()).item()
print(f"Reconstruction error WITH cycle:    {rec_cyc:.4f}")
print(f"Reconstruction error WITHOUT cycle: {rec_nocyc:.4f}")
print("\nWithout cycle loss nothing forces content to survive — compare the two errors.")

## 9. Experiment 2: Identity Loss Effect

Does identity loss help preserve the overall appearance?

In [ ]:
# Test identity: give G_AB a domain B image (should pass through unchanged)
with torch.no_grad():
    id_output = G_AB(test_B).cpu()  # B through A→B gen

fig, axes = plt.subplots(2, n_show, figsize=(12, 3))
for i in range(n_show):
    axes[0, i].imshow((test_B[i].cpu().squeeze() + 1)/2, cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow((id_output[i].squeeze() + 1)/2, cmap="gray")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Input B", fontsize=11, rotation=0, labelpad=45)
axes[1, 0].set_ylabel("G_AB(B)\nshould be\nsame", fontsize=10, rotation=0, labelpad=55)

id_error = F.l1_loss(id_output, test_B.cpu()).item()
plt.suptitle(f"Identity Test: G_AB(domain B image) — L1 error: {id_error:.4f}", fontsize=13)
plt.tight_layout()
plt.show()

print("If identity loss is working, row 2 should look like row 1.")
print("G_AB learns: 'if the input is already in domain B, don't change it.'")

## 10. Experiment 3: Blending Input and Output

A simple pixel blend between the original and its translation (an image cross-fade, not latent interpolation as in L11).

In [ ]:
# Pick one image and interpolate
sample_A = test_A[0:1]
with torch.no_grad():
    translated = G_AB(sample_A)

alphas = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
fig, axes = plt.subplots(1, len(alphas), figsize=(12, 2.5))

for i, alpha in enumerate(alphas):
    blended = (1 - alpha) * sample_A + alpha * translated
    axes[i].imshow((blended.cpu().squeeze() + 1)/2, cmap="gray")
    axes[i].set_title(f"α={alpha}", fontsize=11)
    axes[i].axis("off")

plt.suptitle("Translation Interpolation: α=0 (original) → α=1 (fully translated)", fontsize=13)
plt.tight_layout()
plt.show()

## Key Takeaways

1. **Unpaired translation** — no matching pairs needed, just two collections
2. **Cycle consistency** — A→B→A ≈ A forces content preservation
3. **Two generators, two discriminators** — G_AB, G_BA, D_A, D_B
4. **LSGAN** — more stable than BCE, smoother gradients
5. **Identity loss** — "if already in target domain, don't change it"
6. **ResNet generator** — learns output = input + changes (good for style)
7. **Without cycle loss** — nothing ties output to input; reconstruction error rises
8. **Replay buffer** — prevents D from forgetting old fakes

**Your turn:** set `lambda_id = 0` and retrain. Does the white background of domain B survive the round trip?

**Next: L20 — Satellite to Map + Course Wrap-up**